<a href="https://colab.research.google.com/github/snumryk/TRPA1-ML-benchmark/blob/main/scripts/MolFormer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Імпорт залежностей

In [ ]:
!pip install "transformers==4.57.1" rdkit xgboost scipy scikit-learn -q

import transformers, rdkit, xgboost, torch
print(f"transformers={transformers.__version__}, rdkit={rdkit.__version__}, "
      f"xgboost={xgboost.__version__}, torch={torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.0 MB/s eta 0:00:00
transformers=4.57.1, rdkit=2026.03.3, xgboost=3.2.0, torch=2.11.0+cu128
GPU: Tesla T4


## Mount Drive + Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/trpa1_project'
df = pd.read_csv(f'{DATA_PATH}/trpa1_antagonists.csv')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Mounted at /content/drive
Train: 1324, Val: 160, Test: 161


## RDKit descriptors

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

RDKIT_DESCS = [
    'MolWt', 'MolLogP', 'MolMR', 'TPSA',
    'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds',
    'NumAromaticRings', 'RingCount', 'FractionCSP3',
    'HeavyAtomCount', 'NumAliphaticRings', 'NumSaturatedRings',
    'NumHeteroatoms', 'LabuteASA',
]

def compute_rdkit(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(len(RDKIT_DESCS))
    return np.array([float(getattr(Descriptors, n)(mol)) for n in RDKIT_DESCS], dtype=np.float32)

X_train_rdk = np.vstack(train_df['std_smiles'].apply(compute_rdkit).values)
X_test_rdk  = np.vstack(test_df['std_smiles'].apply(compute_rdkit).values)
y_train = train_df['pchembl_median'].values
y_test  = test_df['pchembl_median'].values

print(f"RDKit features: {X_train_rdk.shape[1]}")
print("RDKit descriptors computed.")

RDKit features: 15
RDKit descriptors computed.


## Load MolFormer + Extract Frozen Embeddings

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

MOLFORMER_NAME = "ibm/MoLFormer-XL-both-10pct"

print(f"Loading {MOLFORMER_NAME}...")
mf_tokenizer = AutoTokenizer.from_pretrained(MOLFORMER_NAME, trust_remote_code=True)
mf_base = AutoModel.from_pretrained(MOLFORMER_NAME, trust_remote_code=True).to('cuda').eval()
print(f"Parameters: {sum(p.numel() for p in mf_base.parameters()):,}")

def extract_embeddings(smiles_list, pooling='cls'):
    embeddings = []
    for smi in tqdm(smiles_list, desc=f"Embedding ({pooling})"):
        tokens = mf_tokenizer(smi, return_tensors="pt", truncation=True,
                              padding=True, max_length=202).to('cuda')
        with torch.no_grad():
            output = mf_base(**tokens)
        hidden = output.last_hidden_state
        if pooling == 'cls':
            emb = hidden[:, 0, :].cpu().numpy().ravel()
        else:
            mask = tokens['attention_mask'].unsqueeze(-1).float().to('cuda')
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            emb = emb.cpu().numpy().ravel()
        embeddings.append(emb)
    return np.vstack(embeddings)

# Extract CLS and Mean Pooling for train + test
X_train_cls  = extract_embeddings(train_df['std_smiles'].tolist(), 'cls')
X_test_cls   = extract_embeddings(test_df['std_smiles'].tolist(), 'cls')
X_train_mean = extract_embeddings(train_df['std_smiles'].tolist(), 'mean')
X_test_mean  = extract_embeddings(test_df['std_smiles'].tolist(), 'mean')

print(f"\nCLS dim: {X_train_cls.shape[1]}, Mean dim: {X_train_mean.shape[1]}")

# Free GPU memory for fine-tuning later
del mf_base
torch.cuda.empty_cache()
print("Embeddings extracted. GPU memory freed.")

Loading ibm/MoLFormer-XL-both-10pct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_molformer_fast.py: 0.00B [00:00, ?B/s]

tokenization_molformer.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- modeling_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/187M [00:00<?, ?B/s]

Parameters: 44,375,040


Embedding (mean): 100%|██████████| 161/161 [00:03<00:00, 48.03it/s]


CLS dim: 768, Mean dim: 768
Embeddings extracted. GPU memory freed.


## Frozen Embeddings + XGBoost/RF

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, matthews_corrcoef
from scipy.stats import spearmanr

SEED = 42
THRESHOLD = 7.0
actuals = y_test

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    rho = spearmanr(y_true, y_pred).correlation
    y_cls = (y_true >= THRESHOLD).astype(int)
    auc = roc_auc_score(y_cls, y_pred)
    mcc = matthews_corrcoef(y_cls, (y_pred >= THRESHOLD).astype(int))
    return rmse, r2, rho, auc, mcc

# Feature combos
features = {
    'MF-CLS(768)':       (X_train_cls, X_test_cls),
    'MF-MeanPool(768)':  (X_train_mean, X_test_mean),
    'MF-CLS+RDKit':      (np.hstack([X_train_cls, X_train_rdk]),
                           np.hstack([X_test_cls, X_test_rdk])),
    'MF-MeanPool+RDKit': (np.hstack([X_train_mean, X_train_rdk]),
                           np.hstack([X_test_mean, X_test_rdk])),
}

print("="*80)
print("FROZEN MolFormer EMBEDDINGS + XGBoost / RF")
print("="*80)
print(f"{'Model':<30} {'RMSE':>6} {'R2':>6} {'Spearman':>9} {'AUC':>6} {'MCC':>6}")
print("-"*80)

frozen_results = []

for feat_name, (Xtr, Xte) in features.items():
    xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                       n_jobs=-1, random_state=SEED)
    xgb.fit(Xtr, y_train)
    preds = xgb.predict(Xte)
    rmse, r2, rho, auc, mcc = evaluate(actuals, preds)
    label = f"XGB + {feat_name}"
    print(f"  {label:<30} {rmse:>6.3f} {r2:>6.3f} {rho:>9.3f} {auc:>6.3f} {mcc:>6.3f}")
    frozen_results.append((label, rmse, r2, rho, auc, mcc))

# RF on combo features
for feat_name in ['MF-CLS+RDKit', 'MF-MeanPool+RDKit']:
    Xtr, Xte = features[feat_name]
    rf = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=SEED)
    rf.fit(Xtr, y_train)
    preds = rf.predict(Xte)
    rmse, r2, rho, auc, mcc = evaluate(actuals, preds)
    label = f"RF + {feat_name}"
    print(f"  {label:<30} {rmse:>6.3f} {r2:>6.3f} {rho:>9.3f} {auc:>6.3f} {mcc:>6.3f}")
    frozen_results.append((label, rmse, r2, rho, auc, mcc))

FROZEN MolFormer EMBEDDINGS + XGBoost / RF
Model                            RMSE     R2  Spearman    AUC    MCC
--------------------------------------------------------------------------------
  XGB + MF-CLS(768)               0.779  0.262     0.555  0.756  0.359
  XGB + MF-MeanPool(768)          0.751  0.314     0.603  0.779  0.413
  XGB + MF-CLS+RDKit              0.720  0.369     0.614  0.788  0.405
  XGB + MF-MeanPool+RDKit         0.692  0.418     0.661  0.807  0.423
  RF + MF-CLS+RDKit               0.736  0.340     0.622  0.802  0.383
  RF + MF-MeanPool+RDKit          0.713  0.383     0.671  0.827  0.378


## Fine-tuning MolFormer (end-to-end)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoConfig
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

# ══════════════════════════════════════════════════════════════
# 1. Завантажити модель з deterministic_eval=True
# ══════════════════════════════════════════════════════════════
config = AutoConfig.from_pretrained(MOLFORMER_NAME, trust_remote_code=True)
config.deterministic_eval = True  # ← КЛЮЧОВИЙ ФІКС

base_model = AutoModel.from_pretrained(
    MOLFORMER_NAME, config=config, trust_remote_code=True
)

# ══════════════════════════════════════════════════════════════
# 2. Кастомна модель, де backbone ЗАВЖДИ в eval mode
# ══════════════════════════════════════════════════════════════
class MolFormerRegressor(nn.Module):
    def __init__(self, backbone, hidden_size=768):
        super().__init__()
        self.backbone = backbone
        self.backbone.eval()  # Backbone ЗАВЖДИ eval
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )
        # Маленька ініціалізація
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.01)
                nn.init.zeros_(m.bias)

    def train(self, mode=True):
        """Override: backbone ЗАВЖДИ залишається в eval mode"""
        super().train(mode)
        self.backbone.eval()  # ← не дозволяємо backbone перейти в train mode
        return self

    def forward(self, input_ids, attention_mask):
        with torch.set_grad_enabled(self.backbone.parameters().__next__().requires_grad):
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_emb).squeeze(-1)

model = MolFormerRegressor(base_model).cuda()
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

# ══════════════════════════════════════════════════════════════
# 3. Dataset (той самий)
# ══════════════════════════════════════════════════════════════
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.smiles = smiles_list
        self.labels = np.array(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_dataset = SMILESDataset(train_df['std_smiles'].tolist(), y_train, mf_tokenizer)
val_dataset = SMILESDataset(val_df['std_smiles'].tolist(), val_df['pchembl_median'].values, mf_tokenizer)
test_dataset = SMILESDataset(test_df['std_smiles'].tolist(), y_test, mf_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ══════════════════════════════════════════════════════════════
# 4. Двоетапний manual training loop
# ══════════════════════════════════════════════════════════════

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].cuda()
            mask = batch['attention_mask'].cuda()
            out = model(ids, mask)
            preds.append(out.cpu().numpy())
            labels.append(batch['labels'].numpy())
    preds = np.concatenate(preds)
    labels = np.concatenate(labels)
    if np.isnan(preds).any():
        return {'loss': float('inf'), 'rmse': float('inf'), 'r2': -1, 'spearman': 0}
    mse = mean_squared_error(labels, preds)
    return {
        'loss': mse,
        'rmse': np.sqrt(mse),
        'r2': r2_score(labels, preds),
        'spearman': spearmanr(labels, preds).correlation
    }

def train_epoch(model, loader, optimizer, max_grad_norm=1.0):
    model.train()  # наш override тримає backbone в eval!
    total_loss = 0
    for batch in loader:
        ids = batch['input_ids'].cuda()
        mask = batch['attention_mask'].cuda()
        labels = batch['labels'].cuda()

        preds = model(ids, mask)
        loss = nn.MSELoss()(preds, labels)

        if torch.isnan(loss):
            print("  ⚠️ NaN loss detected, skipping batch")
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ── Stage 1: Тільки head (backbone заморожений) ──
print("=" * 60)
print("Stage 1: Training head only (3 epochs, lr=1e-3)")
print("=" * 60)

for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=0.01)

for epoch in range(3):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_metrics = evaluate(model, val_loader)
    print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f} | "
          f"val_rmse={val_metrics['rmse']:.4f} | val_r2={val_metrics['r2']:.4f}")

# ── Stage 2: End-to-end (backbone розморожений) ──
print("=" * 60)
print("Stage 2: End-to-end fine-tuning (max 50 epochs, patience=5)")
print("=" * 60)

for p in model.backbone.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 5e-6},
    {'params': model.head.parameters(), 'lr': 1e-4},
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(50):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_metrics = evaluate(model, val_loader)
    scheduler.step()

    print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f} | "
          f"val_rmse={val_metrics['rmse']:.4f} | "
          f"val_r2={val_metrics['r2']:.4f} | "
          f"val_spearman={val_metrics['spearman']:.4f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()
                      if 'head' in k}  # Зберігаємо тільки head (уникаємо non-contiguous)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f"  Early stopping at epoch {epoch+1}")
            break

# Завантажити найкращі ваги head
if best_state:
    model.load_state_dict(best_state, strict=False)

# ── Фінальна оцінка ──
test_metrics = evaluate(model, test_loader)
print(f"\n{'='*60}")
print(f"TEST: RMSE={test_metrics['rmse']:.4f} | R²={test_metrics['r2']:.4f} | "
      f"Spearman={test_metrics['spearman']:.4f}")

Total params: 44,573,697
Stage 1: Training head only (3 epochs, lr=1e-3)
  Epoch 1: train_loss=11.7073 | val_rmse=0.8003 | val_r2=0.1552
  Epoch 2: train_loss=0.6546 | val_rmse=0.6408 | val_r2=0.4584
  Epoch 3: train_loss=0.5149 | val_rmse=0.7036 | val_r2=0.3470
Stage 2: End-to-end fine-tuning (max 50 epochs, patience=5)
  Epoch 1: train_loss=0.4632 | val_rmse=0.6283 | val_r2=0.4794 | val_spearman=0.6700
  Epoch 2: train_loss=0.3586 | val_rmse=0.6160 | val_r2=0.4996 | val_spearman=0.7032
  Epoch 3: train_loss=0.3323 | val_rmse=0.6409 | val_r2=0.4582 | val_spearman=0.7049
  Epoch 4: train_loss=0.3155 | val_rmse=0.6112 | val_r2=0.5074 | val_spearman=0.7101
  Epoch 5: train_loss=0.2807 | val_rmse=0.6184 | val_r2=0.4957 | val_spearman=0.7070
  Epoch 6: train_loss=0.2757 | val_rmse=0.6097 | val_r2=0.5098 | val_spearman=0.7090
  Epoch 7: train_loss=0.2535 | val_rmse=0.6214 | val_r2=0.4908 | val_spearman=0.7162
  Epoch 8: train_loss=0.2509 | val_rmse=0.5936 | val_r2=0.5353 | val_spearman=0.71

## Test Results + Full Comparison

In [ ]:
# ══════════════════════════════════════════════════════════════
# Test Results + Full Comparison
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import roc_auc_score, matthews_corrcoef

# Prediction з manual-trained моделі (вже на GPU)
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].cuda()
        mask = batch['attention_mask'].cuda()
        out = model(ids, mask)
        all_preds.append(out.cpu().numpy())
        all_labels.append(batch['labels'].numpy())

preds_ft = np.concatenate(all_preds)
actuals = np.concatenate(all_labels)

# Перевірка на NaN
nan_count = np.isnan(preds_ft).sum()
if nan_count > 0:
    print(f"⚠️  {nan_count}/{len(preds_ft)} NaN predictions in test set")

# Метрики (використовуємо ту ж логіку що в клітинці 5)
finite_mask = np.isfinite(preds_ft)
ft_success = finite_mask.sum() >= 2

if ft_success:
    p_clean = preds_ft[finite_mask]
    a_clean = actuals[finite_mask]
    rmse_ft = np.sqrt(mean_squared_error(a_clean, p_clean))
    r2_ft = r2_score(a_clean, p_clean)
    rho_ft = spearmanr(a_clean, p_clean).correlation
    y_cls = (a_clean >= THRESHOLD).astype(int)
    auc_ft = roc_auc_score(y_cls, p_clean)
    mcc_ft = matthews_corrcoef(y_cls, (p_clean >= THRESHOLD).astype(int))
else:
    rmse_ft = r2_ft = rho_ft = auc_ft = mcc_ft = None

# ── Таблиця порівняння ──
print("=" * 85)
print("COMPLETE COMPARISON: ChemBERTa vs MolFormer vs Classical vs GNN")
print("=" * 85)
print(f"{'Model':<35} {'RMSE':>6} {'R2':>6} {'Spearman':>9} {'AUC':>6}  {'Method'}")
print("-" * 85)

all_rows = [
    ("RF (Morgan 2048)",              0.795, 0.232, 0.552, 0.795, "Fingerprint"),
    ("XGB (Morgan 2048)",             0.781, 0.257, 0.547, 0.790, "Fingerprint"),
    ("RF (15 RDKit)",                 0.724, 0.363, 0.624, 0.800, "Descriptors"),
    ("D-MPNN (graph only)",           0.823, 0.176, 0.554, 0.761, "GNN"),
    ("D-MPNN+RDKit (ES)",             0.763, 0.292, 0.621, 0.810, "GNN+Desc"),
    ("XGB+CB-CLS(384)",               0.735, 0.344, 0.642, 0.806, "CB frozen"),
    ("XGB+CB-CLS+RDKit",              0.706, 0.395, 0.672, 0.811, "CB frozen+Desc"),
    ("ChemBERTa fine-tuned",          0.787, 0.247, 0.497, 0.739, "CB fine-tuned"),
]

for name, rmse, r2, rho, auc, method in all_rows:
    print(f"  {name:<35} {rmse:>6.3f} {r2:>6.3f} {rho:>9.3f} {auc:>6.3f}  {method}")

print()
for name, rmse, r2, rho, auc, mcc in frozen_results:
    print(f"  {name:<35} {rmse:>6.3f} {r2:>6.3f} {rho:>9.3f} {auc:>6.3f}  MF frozen")

if ft_success:
    print(f"  {'MolFormer fine-tuned':<35} {rmse_ft:>6.3f} {r2_ft:>6.3f} {rho_ft:>9.3f} {auc_ft:>6.3f}  MF fine-tuned ✓")
else:
    print(f"  {'MolFormer fine-tuned':<35} {'— FAILED (NaN) —':>30}  MF fine-tuned ✗")

print("-" * 85)

# ── Висновки ──
if ft_success:
    print(f"\nMolFormer fine-tuned:")
    print(f"  vs ChemBERTa fine-tuned:       R² {r2_ft:.3f} vs 0.247 (Δ {r2_ft-0.247:+.3f})")
    print(f"  vs best frozen (XGB+CB+RDKit): R² {r2_ft:.3f} vs 0.395 (Δ {r2_ft-0.395:+.3f})")

    best_frozen_r2 = max(r[2] for r in frozen_results)
    print(f"  vs best MF frozen:             R² {r2_ft:.3f} vs {best_frozen_r2:.3f} (Δ {r2_ft-best_frozen_r2:+.3f})")

    if r2_ft > 0.395:
        print("\n✓ MolFormer fine-tuning BEATS all frozen approaches!")
    elif r2_ft > 0.247:
        print("\n~ MolFormer FT > ChemBERTa FT, but < frozen embeddings + XGBoost")
    else:
        print("\n✗ Both fine-tuned transformers underperform frozen embeddings")
else:
    print("\n⚠️  MolFormer end-to-end fine-tuning FAILED (NaN outputs).")
    print("   Known issue: linear attention + random feature maps is numerically")
    print("   unstable in training mode (IBM/molformer GitHub issue #22).")
    print("\n   CONCLUSION: Frozen MolFormer embeddings + XGBoost = best strategy")
    print("   for small datasets. This is a valid and publishable finding.")

COMPLETE COMPARISON: ChemBERTa vs MolFormer vs Classical vs GNN
Model                                 RMSE     R2  Spearman    AUC  Method
-------------------------------------------------------------------------------------
  RF (Morgan 2048)                     0.795  0.232     0.552  0.795  Fingerprint
  XGB (Morgan 2048)                    0.781  0.257     0.547  0.790  Fingerprint
  RF (15 RDKit)                        0.724  0.363     0.624  0.800  Descriptors
  D-MPNN (graph only)                  0.823  0.176     0.554  0.761  GNN
  D-MPNN+RDKit (ES)                    0.763  0.292     0.621  0.810  GNN+Desc
  XGB+CB-CLS(384)                      0.735  0.344     0.642  0.806  CB frozen
  XGB+CB-CLS+RDKit                     0.706  0.395     0.672  0.811  CB frozen+Desc
  ChemBERTa fine-tuned                 0.787  0.247     0.497  0.739  CB fine-tuned

  XGB + MF-CLS(768)                    0.779  0.262     0.555  0.756  MF frozen
  XGB + MF-MeanPool(768)               0.751  